In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2005539834499359
epoch:  1, loss: 0.19998587667942047
epoch:  2, loss: 0.16897884011268616
epoch:  3, loss: 0.12655693292617798
epoch:  4, loss: 0.09402631223201752
epoch:  5, loss: 0.0825452208518982
epoch:  6, loss: 0.06671982258558273
epoch:  7, loss: 0.021789928898215294
epoch:  8, loss: 0.01151354145258665
epoch:  9, loss: 0.007746642455458641
epoch:  10, loss: 0.006275742314755917
epoch:  11, loss: 0.0054555414244532585
epoch:  12, loss: 0.004945608787238598
epoch:  13, loss: 0.004586147144436836
epoch:  14, loss: 0.004128177184611559
epoch:  15, loss: 0.003760578343644738
epoch:  16, loss: 0.003523392602801323
epoch:  17, loss: 0.0032858559861779213
epoch:  18, loss: 0.0030873343348503113
epoch:  19, loss: 0.0029253745451569557
epoch:  20, loss: 0.0027897574473172426
epoch:  21, loss: 0.0026594889350235462
epoch:  22, loss: 0.0025538867339491844
epoch:  23, loss: 0.0024650730192661285
epoch:  24, loss: 0.0023824896197766066
epoch:  25, loss: 0.0023002822417765

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9659268260002136
Test metrics:  R2 = 0.6670517921447754


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="interpolate",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.364336758852005
epoch:  1, loss: 0.364336758852005
epoch:  2, loss: 0.364336758852005
epoch:  3, loss: 0.364336758852005
epoch:  4, loss: 0.364336758852005
epoch:  5, loss: 0.364336758852005
epoch:  6, loss: 0.364336758852005
epoch:  7, loss: 0.364336758852005
epoch:  8, loss: 0.364336758852005
epoch:  9, loss: 0.364336758852005
epoch:  10, loss: 0.364336758852005
epoch:  11, loss: 0.364336758852005
epoch:  12, loss: 0.364336758852005
epoch:  13, loss: 0.364336758852005
epoch:  14, loss: 0.364336758852005
epoch:  15, loss: 0.364336758852005
epoch:  16, loss: 0.364336758852005
epoch:  17, loss: 0.364336758852005
epoch:  18, loss: 0.364336758852005
epoch:  19, loss: 0.364336758852005
epoch:  20, loss: 0.364336758852005
epoch:  21, loss: 0.364336758852005
epoch:  22, loss: 0.364336758852005
epoch:  23, loss: 0.364336758852005
epoch:  24, loss: 0.364336758852005
epoch:  25, loss: 0.364336758852005
epoch:  26, loss: 0.364336758852005
epoch:  27, loss: 0.364336758852005
ep

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -31953.150390625
Test metrics:  R2 = -33111.26171875


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="bisect",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1275000423192978
epoch:  1, loss: 0.054539963603019714
epoch:  2, loss: 0.040493592619895935
epoch:  3, loss: 0.0323735736310482
epoch:  4, loss: 0.02557680942118168
epoch:  5, loss: 0.02251257188618183
epoch:  6, loss: 0.01752891205251217
epoch:  7, loss: 0.014117744751274586
epoch:  8, loss: 0.012963629327714443
epoch:  9, loss: 0.011299471370875835
epoch:  10, loss: 0.010287942364811897
epoch:  11, loss: 0.008918493054807186
epoch:  12, loss: 0.0080709895119071
epoch:  13, loss: 0.007684021256864071
epoch:  14, loss: 0.007147683296352625
epoch:  15, loss: 0.006800293922424316
epoch:  16, loss: 0.0066056568175554276
epoch:  17, loss: 0.006412372924387455
epoch:  18, loss: 0.006204608362168074
epoch:  19, loss: 0.006011165678501129
epoch:  20, loss: 0.005852797534316778
epoch:  21, loss: 0.005738477688282728
epoch:  22, loss: 0.005649338010698557
epoch:  23, loss: 0.005573296453803778
epoch:  24, loss: 0.005526227876543999
epoch:  25, loss: 0.005441625136882067
epoc

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9190704822540283
Test metrics:  R2 = 0.5939576625823975
